# 单个国家的校对，美国，巴西，动物农作物

**步骤：**
1. 3个函数定义，函数一：fao规范化；函数二:历史比例r存储出来;函数三：校对;函数四：统一单位
2. 输入种类的一一对应关系params字典，下载fao数据，执行函数一
3. 执行函数二得到r，保存起来
4. 执行函数四，统一原始数据单位与FAO的一致
5. 执行函数三，得到最终校对结果

# 函数定义
---

In [5]:
import numpy as np
def convert_to_number(value):
    # 如果值已经是数值类型，直接返回该值
    if isinstance(value, (int, float)):
        return value
    
    try:
        # 尝试将值转换为浮点数
        return float(value)
    except (ValueError, TypeError):
        # 如果转换失败，返回0
        return 0
    
# 函数一、二、三定义
import pandas as pd 
# 函数一
# 规范化fao数据，注意单位,单位要统一成FAO一样的，然后FAO动物的要统一成An，
# 输入所有年份的fao原始数据,输出分年份规范化好的fao数据
def fao_standard(fao_data,params,target_path):
    # params是字典，键是fao数据的动物种类名称，值是键对应的要转化的种类名称，所有值是列表则证明要将这两项相加
    # 确保数值列可以做四则运算
    value_name = "Value" if "Value" in fao_data.columns else "value"
    fao_data[value_name] = fao_data[value_name].astype(str)
    fao_data[value_name] = fao_data[value_name].str.replace(',', '')
    fao_data[value_name] = pd.to_numeric(fao_data[value_name], errors='coerce')

    # 单位转化
    fao_data.loc[fao_data['Unit'] == '1000 An', 'Value'] *= 1000
    fao_data.loc[fao_data['Unit'] == '1000 An', 'Unit'] = 'An'

    for year,group in fao_data.groupby("Year"):
        # 一年一年来，搞完保存

        # fao_data数据只有Item列需要,用replace替换
        for item in params:
            if isinstance(item,str):
                group["Item"] = group['Item'].replace(item,params[item])
            elif isinstance(item,tuple):
                # 先相加，按照年份，然后形成新的一个值
                all = 0
                for i in range(0,len(item)):
                    if group[group['Item']==item[i]].shape[0]>0:

                        all += group[group['Item']==item[i]][value_name].values[0]
                
                for i in range(0,len(item)):
                    
                    if group[group['Item']==item[i]].shape[0] > 0:
                        group.loc[group["Item"]==item[i],value_name] = all
                            
                        group["Item"] = group['Item'].replace(item[i],params[item])
                        break

                
                
                # for i in range(1,len(item)):
                #     group.loc[group["Item"]==item[0],value_name] += group[group['Item']==item[i]][value_name].values[0]
                    
                # group["Item"] = group['Item'].replace(item[0],params[item]) 
                    
            else:
                print("params输入格式错误,错误的键为{}".format(item))

        # 保存
        group.to_csv(target_path+str(year)+".csv")


# 函数二
# 美国、巴西,单个国家的校对
def cal_r(data,data_fao,Year,data_fao_item="Item"):
    r = data.iloc[:,0:3] # 存储比例
    r['year'] = Year

    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"
    
    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            r[animal] = ''
            # 首先转化列里面数值保证能进行四则运算
            r[animal] = r[animal].astype(str)
            r[animal] = r[animal].str.replace(',', '')  # remove commas
            r[animal] = pd.to_numeric(r[animal], errors='coerce').fillna(0)

            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')
            data_fao[value_name] = data_fao[value_name].astype(str)
            data_fao[value_name] = data_fao[value_name].str.replace(',', '')
            data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

            # 首先计算data里面每个县占比
            country_total = data[animal].sum()
            if country_total != 0:
                proportions = data[animal] / country_total
                r[animal] = proportions
            else:
                continue

    return r



# 函数三
# 与FAO数据校对,注意单位
# 美国、巴西,单个国家的校对
def single_proofread_r(data,data_fao,data_fao_item="Item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量

    
    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"
    

    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            
            # # 首先转化列里面数值保证能进行四则运算
            # r[animal] = r[animal].astype(str)
            # r[animal] = r[animal].str.replace(',', '')  # remove commas
            # r[animal] = pd.to_numeric(r[animal], errors='coerce').fillna(0)

            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')
            data_fao[value_name] = data_fao[value_name].astype(str)
            data_fao[value_name] = data_fao[value_name].str.replace(',', '')
            data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

            # 首先计算data里面每个县占比
            country_total = data[animal].sum()
            if country_total != 0:
                proportions = data[animal] / country_total
              
            else:
                proportions = np.random.dirichlet(np.ones(data[animal].shape[0]), size=1).flatten()  
            print(f"该国该种类：{data[animal].shape[0]}")
                #  # 找出最近的年份，其中 'animal' 列的值的总和大于等于 0
                # for offset in range(1, max(Year - 1980, 2021 - Year) + 1):
                #     recent_year = Year - offset if Year - offset >= 1980 else Year + offset
                #     if recent_year <= 2021 and r[ r['year']==recent_year][animal].sum() >= 0:
                #         proportions = r[r['year']==recent_year][animal]
                #         break

            # 如果FAO数据缺失那么就不进行校对
            fao_value = data_fao[data_fao[data_fao_item]==animal][value_name].values[0]

            # if abs(fao_value-data[animal].sum()) > 0.2*max(fao_value,data[animal].sum()):
            #     count_error += 1
            #     if country_total==0 and fao_value != 0:
            #         count_zero += 1
                # else:
                #     data[animal] = proportions*fao_value
                # fao总量与国家总量差别太大的不要
                    # data["标记"] += "种类："+animal+","+"FAO总量："+str(fao_value)+","+"国家总量："+str(data[animal].sum())+"; "

            # print("年份：{}，种类为{}的fao总量为{}，对应国家数据总量{}".format(Year,animal,fao_value,country_total))

            # 检查value是否为数值类型
            fao_value = convert_to_number(fao_value)
            # 如果是数值类型，检查是否大于等于0
            if fao_value > 0:
                if sum(proportions)==0:
                    proportions = np.random.dirichlet(np.ones(data[animal].shape[0]), size=1).flatten()  

                    # proportions = np.random.random(size=len(proportions))
                   
                    
                data[animal] = proportions*fao_value
            print(f'国家：{data_fao.Area.unique()}的种类：{animal}，校对成功，校对后为：{data[animal].sum()}，fao值为{fao_value}')
    
    
    return data

# 函数四 单位转化
# 转化原始数据单位，确保与FAO统一
def unit_conversion(data,params):
    # params是字典，键是列名一部分(如area)，值是转化比例scale
    for col_name in params.keys():
        for item in data.columns:
            if isinstance(col_name,str):
                if col_name in item:
                    data[item] = data[item].astype(str)
                    data[item] = data[item].str.replace(',', '')
                    data[item] = pd.to_numeric(data[item], errors='coerce')
                    data[item] *= params[col_name]
            elif isinstance(col_name,tuple):
                for i in range(len(col_name)):
                    if col_name[i] in item:
                        data[item] = data[item].astype(str)
                        data[item] = data[item].str.replace(',', '')
                        data[item] = pd.to_numeric(data[item], errors='coerce')
                        data[item] *= params[col_name]
            else:
                print("params输入格式错误,错误的键为{}".format(item))

    return data


# 输入
---

In [7]:
# 巴西农作物

params = {
    "Pineapples":"Pineapple*",
    "Avocados":"Abacate",
    "Seed cotton, unginned":"Herbaceous cotton (seed)",
    "Green garlic":"Garlic",
    "Groundnuts, excluding shelled":"Peanuts (in shell)",
    "Rice":"Rice",
    "Oats":"Oats (grain)",
    "Olives":"Olive",
    "Bananas":"Banana (bunch)",
    "Sweet potatoes":"Sweet potatoes",
    "potatos":"Batata-inglesa",
    "Natural rubber in primary forms":"Rubber (coagulated latex)",
    "Cocoa beans":"Cocoa beans",
    "Coffee, green":"Coffee (beans) Total",
    "Sugar cane":"Sugarcane",
    "Persimmons":"Persimmon",
    "Cashew nuts, in shell":"Cashew nuts",
    "Onions and shallots, dry (excluding dehydrated)":"Onion",
    "Rye":"Rye (grain)",
    "Barley":"Barley (grain)",
    "Tea leaves":"Tea(leafy green)",
    "Oil palm fruit":"Palm oil(coconut bunch)",
    "Maté leaves":"Yerba mate (green leaf)",
    "Peas, dry":"Peas (grain)",
    "Broad beans and horse beans, dry":"Fava beans",
    "Beans, dry":"Beans (beans)",
    "Figs":"Fig",
    "Unmanufactured tobacco":"Tobacco",
    "Sunflower seed":"Sunflower (grain)",
    "Jute, raw or retted":"Jute (fiber)",
    "Oranges":"Orange",
    "Lemons and limes":"Lemon",
    "Linseed":"Flax (seed)",
    "Apples":"Apple",
    "Kenaf, and other textile bast fibres, raw or retted":"Mallow (fiber)",
    "Papayas":"Papaya",
    "Castor oil seeds":"Mormon(bag)",
    "Cassava, fresh":"Cassava",
    "Quinces":"Marble",
    "Watermelons":"Watermelon",
    "Cantaloupes and other melons":"Melon",
    "Maize (corn)":"Corn (in grain)",
    "Walnuts, in shell":"Walnut(dried fruit)",
    "Oil palm fruit":"Palm",
    "Pears":"Pear",
    "Peaches and nectarines":"Peach",
    "Pepper (Piper spp.), raw":"Black pepper",
    "Ramie, raw or retted":"Rami (fiber)",
    "Sisal, raw":"Sisal or agave (fiber)",
    "Soya beans":"Soybeans",
    "Sorghum":"Sorghum (grain)",
    "Tangerines, mandarins, clementines":"Tangerine",
    "Tomatoes":"Tomato",
    "Wheat":"Wheat (grain)",
    "Triticale":"Triticale (grain)",
    "Tung nuts":"Tungue(fruto seco)",
    "Grapes":"Grape"
}

fao_target_path = '校对总量全（新）/巴西/FAO/' # 规范化后的fao数据的保存路径
fao_data_path =  '校对总量全（新）/巴西/FAO/农作物/1974.csv'  # fao原始数据的存放路径
data_path = '巴西/农作物_ok/' # 需要校对的国家数据存放路径
r_path = '校对总量全（新）/巴西/FAO/比例县.csv' # 历史比例存放路径
data_fao_ok = '校对总量全（新）/巴西/农作物_fao_ok/' # 校对完成后的数据存放路径

# 校对国家的开始与结束年份
year_begin = 1974
year_end = 1974



In [81]:
# 美国动物
params = {
    "Raw milk of cattle":"cattle_cow_milk",
    ("Meat of cattle with the bone, fresh or chilled","Meat of buffalo, fresh or chilled"):"cattle_cow_beef",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"hogs",
    ("Sheep","Goats"):"sheep_goats",
    "Hen eggs in shell, fresh":"layers",
    "Meat of chickens, fresh or chilled":"broilers"
}

fao_target_path = 'D:/中科院数据下载/USDAquickstats/动物_ok/fao数据/' # 规范化后的fao数据的保存路径
# fao_data_path =  'D:/中科院数据下载/巴西/动物_ok/fao/FAOSTAT_data_en_1-23-2024.csv'  # fao原始数据的存放路径
data_path = 'USDAquickstats/动物_ok/' # 需要校对的国家数据存放路径
r_path = 'USDAquickstats/动物_ok/fao数据/比例县.csv' # 历史比例存放路径
data_fao_ok = 'USDAquickstats/动物_fao_ok/' # 校对完成后的数据存放路径

# 校对国家的开始与结束年份
year_begin = 1997
year_end = 1997


In [98]:
# 巴西动物
params = {
    "Raw milk of cattle":"milk_cattle",
    "Meat of cattle with the bone, fresh or chilled":"beef_cattle",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"Pork - total",
    ('Sheep','Goats'):"Sheep_Goat",
    "Hen eggs in shell, fresh":"Layers",
    "Meat of chickens, fresh or chilled":"Broilers",
    "Horse meat, fresh or chilled":'Horse'
} # fao种类与各个国家官网下载的种类一一对应，左边写fao的种类名称，右边写国家官网种类名称


fao_target_path = 'D:/中科院数据下载/巴西/动物_ok/fao/' # 规范化后的fao数据的保存路径
fao_data_path =  'D:/中科院数据下载/巴西/动物_ok/fao/FAOSTAT_data_en_1-23-2024.csv'  # fao原始数据的存放路径
data_path = 'D:/中科院数据下载/巴西/动物_ok/' # 需要校对的国家数据存放路径
r_path = 'D:/中科院数据下载/巴西/动物_ok/fao/比例县.csv' # 历史比例存放路径
data_fao_ok = 'D:/中科院数据下载/巴西/动物_fao_ok/' # 校对完成后的数据存放路径

# 校对国家的开始与结束年份
year_begin = 1974
year_end = 2022

# 单位转化参数
params_unit = {
    
}

In [4]:
# # 美国农作物
# params = {
#     'Barley':'PRODUCTION_BARLEY',
#     'Beans, dry':'PRODUCTION_BEANS',
#     'Rape or colza seed':'PRODUCTION_CANOLA',
#     'Chick peas, dry':'PRODUCTION_CHICKPEAS',
#     'Green corn (maize)':'PRODUCTION_CORN',
#     'Seed cotton, unginned':'PRODUCTION_COTTON',
#     'Flax, processed but not spun':'PRODUCTION_FLAXSEED',
#     'Lentils, dry':'PRODUCTION_LENTILS',
#     'Mustard seed':'PRODUCTION_MUSTARD',
#     'Oats':'PRODUCTION_OATS',
#     'Groundnuts, excluding shelled':'PRODUCTION_PEANUTS',
#     'Peas, dry':'PRODUCTION_PEAS',
#     'Rice':'PRODUCTION_RICE',
#     'Rye':'PRODUCTION_RYE',
#     'Safflower seed':'PRODUCTION_SAFFLOWER',
#     'Sorghum':'PRODUCTION_SORGHUM',
#     'Soya beans':'PRODUCTION_SOYBEANS',
#     'Sugar beet':'PRODUCTION_SUGARBEETS',
#     'Sugar cane':'PRODUCTION_SUGARCANE',
#     'Sunflower seed':'PRODUCTION_SUNFLOWER',
#     'Unmanufactured tobacco':'PRODUCTION_TOBACCO',
#     'Wheat':'PRODUCTION_WHEAT',
    
# } # fao种类与各个国家官网下载的种类一一对应,每个种类都要写,本质是改名，统一名称


# fao_target_path = 'D:/中科院数据下载/USDAquickstats/农作物_ok/FAO/' # 规范化后的fao数据的保存路径
# fao_data_path =  'D:/中科院数据下载/USDAquickstats/农作物_ok/FAO/FAO总.csv'  # fao原始数据的存放路径
# data_path = 'D:/中科院数据下载/USDAquickstats/农作物_ok/' # 需要校对的国家数据存放路径
# r_path = 'D:/中科院数据下载/USDAquickstats/农作物_ok/FAO/比例县.csv' # 历史比例存放路径
# data_fao_ok = 'D:/中科院数据下载/USDAquickstats/农作物_fao_ok/' # 校对完成后的数据存放路径

# # 校对国家的开始与结束年份
# year_begin = 1961
# year_end = 2021

# # 单位转化参数
# params_unit = {
#     "BARLEY":0.02177, # 单位BU，麦子 ,基于平均重量为48磅
#     ("BEANS","CHICKPEAS","LENTILS","PEAS","RICE"):0.045359, # beans单位是CWT美担，要转化成t，1英担等于0.050802吨
#     ("CANOLA","MUSTARD","PEANUTS","SAFFLOWER","SUNFLOWER","TOBACCO"):0.0004536, # 单位LB
#     ("CORN","FLAXSEED","RYE","SORGHUM"):0.02540, # 单位BU,玉米,基于平均重量是56磅
#     "COTTON":0.2177, # 单位480LB BALES
#     "OATS":0.01452, # 单位BU,燕麦,基于平均重量是32磅
#     ("SOYBEANS","WHEAT"):0.02722 # 单位BU，大豆，基于平均重量60磅
# } # 单位不一样的种类都要写




In [9]:
# 澳大利亚农作物
params = {
    'Barley':'Cereal crops - Barley for grain - Production (t)',
    'Rape or colza seed':'Other crops - Oilseeds - Canola - Production (t)',
    'Chick peas, dry':'Other crops - Pulses and legumes - Chickpeas - Production (t)',
    'Lentils, dry':'Other crops - Pulses and legumes - Lentils - Production (t)',
    'Oats':'Cereal crops - Oats for grain - Production (t)',
    'Groundnuts, excluding shelled':'PRODUCTION_PEANUTS',
    'Rice':'Cereal crops - Rice for grain - Production (t)',
    'Sorghum':'Cereal crops - Sorghum for grain - Production (t)',
    'Sugar cane':'Other crops - Sugar cane - Cut for crushing - Production (t)',
    'Wheat':'Cereal crops - Wheat for grain - Production (t)',
    'Maize (corn)':'Cereal crops - Maize for grain - Production (t)',
    'Lupins':'Other crops - Pulses and legumes - Lupins - Production (t)',
    'Tangerines, mandarins, clementines':'Fruit and nuts - Citrus fruit - Mandarins - Production (t)',
    'Oranges':'Fruit and nuts - Citrus fruit - Oranges - Production (t)',
    'Cherries':'Fruit and nuts - Stone fruit - Cherries - Production (t)',
    'Apples':'Fruit and nuts - Other orchard fruit - Apples - Production (t)',
    'Avocados':'Fruit and nuts - Other orchard fruit - Avocados - Production (t)',
    'Mangoes, guavas and mangosteens':'Fruit and nuts - Other orchard fruit - Mangoes - Production (t)',
    'Olives':'Fruit and nuts - Other orchard fruit - Olives - Production (t)',
    'Pears':'Fruit and nuts - Other orchard fruit - Pears (including Nashi) - Production (t)',
    'Almonds, in shell':'Fruit and nuts - Nuts - Almonds - Production (t) (f)',
    'Other nuts (excluding wild edible nuts and groundnuts), in shell, n.e.c.':'Fruit and nuts - Nuts - Macadamias - Production (t) (g)',
    'Strawberries':'Fruit and nuts - Berry fruit - Strawberries - Production (t)',
    'Bananas':'Fruit and nuts - Plantation fruit - Bananas - Production (t)',
    'Pineapples':'Fruit and nuts - Plantation fruit - Pineapples - Production (t)',
    'Grapes':'Fruit and nuts - Grapes - Total - Production (t)',
    'Other beans, green':'Vegetables - Beans (including french and runner) - Production (kg)',
    'Cabbages':'Vegetables - Cabbages - Production (t)',
    'Chillies and peppers, green (Capsicum spp. and Pimenta spp.)':'Vegetables - Capsicums (excluding chillies) - Production (kg)',
    'Carrots and turnips':'Vegetables - Carrots - Production (t)',
    'Cucumbers and gherkins':'Vegetables - Cucumbers - Production (t)',
    'Lettuce and chicory':'Vegetables - Lettuces - Production (kg)',
    'Cantaloupes and other melons':'Vegetables - Melons - Production (t) (i)',
    'Mushrooms and truffles':'Vegetables - Mushrooms - Production (kg)',
    'Onions and shallots, dry (excluding dehydrated)':'Vegetables - Onions - Production (t)',
    'Potatoes':'Vegetables - Potatoes - Production (t)',
    'Pumpkins, squash and gourds':'Vegetables - Pumpkins - Production (t)',
    'Green corn (maize)':'Vegetables - Sweet corn - Production (t)',
    'Tomatoes':'Vegetables - Tomatoes - Production (t)'
} # fao种类与各个国家官网下载的种类一一对应,每个种类都要写,本质是改名，统一名称
# 左边fao，右边官网


fao_target_path = 'D:/中科院数据下载/澳大利亚/' # 规范化后的fao数据的保存路径
fao_data_path =  'D:/中科院数据下载/澳大利亚/FAO总.csv'  # fao原始数据的存放路径
data_path = 'D:/中科院数据下载/澳大利亚/' # 需要校对的国家数据存放路径
r_path = 'D:/中科院数据下载/澳大利亚/比例县.csv' # 历史比例存放路径
data_fao_ok = 'D:/中科院数据下载/澳大利亚/农作物_fao_ok/' # 校对完成后的数据存放路径

# 校对国家的开始与结束年份
year_begin = 2021
year_end = 2021





In [6]:
# 欧盟农作物，拆成一个一个的国家
params = {
    'Other stimulant, spice and aromatic crops, n.e.c.':'Harvested production (1000 t)_Aromatic, medicinal and culinary plants',
    'Barley':'Harvested production (1000 t)_Barley',
    'Oranges':'Harvested production (1000 t)_Citrus fruits',
    'Wheat':'Harvested production (1000 t)_Wheat and spelt',
    'Seed cotton, unginned':'Harvested production (1000 t)_Cotton seed',
    
    'True hemp, raw or retted':'Harvested production (1000 t)_Hemp',
    'Linseed':'Harvested production (1000 t)_Linseed (oilflax)',
    'Olives':'Harvested production (1000 t)_Olives',
    'Potatoes':'Harvested production (1000 t)_Potatoes (including seed potatoes)',
    'Sorghum':'Harvested production (1000 t)_Sorghum',
    'Soya beans':'Harvested production (1000 t)_Soya',
    'Sugar beet':'Harvested production (1000 t)_Sugar beet (excluding seed)',
    'Sunflower seed':'Harvested production (1000 t)_Sunflower seed',
    'Lupins':'Harvested production (1000 t)_Sweet lupins',
    'Unmanufactured tobacco':'Harvested production (1000 t)_Tobacco',
    'Triticale':'Harvested production (1000 t)_Triticale',
    'Rye':'Harvested production (1000 t)_Rye',
    'Oats':'Harvested production (1000 t)_Oats',
    'Rice':'Harvested production (1000 t)_Rice',
    'Peas, dry':'Harvested production (1000 t)_Field peas',
    'Other fibre crops, raw, n.e.c.':'Harvested production (1000 t)_Other fibre crops n.e.c.',
    'Other oil seeds, n.e.c.':'Harvested production (1000 t)_Other oilseed crops n.e.c.',
    'Grapes':'Harvested production (1000 t)_Grapes',
    
    'Broad beans and horse beans, dry':'Harvested production (1000 t)_Broad and field beans',
    'Flax, raw or retted':'Harvested production (1000 t)_Fibre crops',
    
    ('Watermelons','Strawberries'):'Harvested production (1000 t)_Fruits, berries and nuts (excluding citrus fruits, grapes and strawberries)',  #  Strawberries
    'Maize (corn)':'Harvested production (1000 t)_Green maize',
    'Hop cones':'Harvested production (1000 t)_Hops',
    'Other beans, green':'Harvested production (1000 t)_Leguminous plants harvested green',
    
    'Millet':'Harvested production (1000 t)_Other cereals n.e.c. (buckwheat, millet, canary seed, etc.)',
    'Rape or colza seed':'Harvested production (1000 t)_Rape, turnip rape, sunflower seeds and soya',
    'Cereals n.e.c.':'Harvested production (1000 t)_Spring cereal mixtures (mixed grain other than maslin)'
} # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称

fao_target_path = '校对总量全（新）/欧盟/FAO/农作物/' # 规范化后的fao数据的保存路径
fao_data_path = 'eurostat/农作物_ok/FAO/FAOSTAT_data_en_1-28-2024.csv'  # fao原始数据的存放路径
data_path = 'eurostat/农作物_ok/' # 需要校对的国家数据存放路径
r_path = '校对总量全（新）/欧盟/FAO/比例县.csv' # 历史比例存放路径
data_fao_ok = '校对总量全（新）/欧盟/' # 校对完成后的数据存放路径

# 校对国家的开始与结束年份
year_begin = 1980
year_end = 1980

In [2]:
# 欧盟农作物，拆成一个一个的国家
params = {
    'Other stimulant, spice and aromatic crops, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Aromatic, medicinal and culinary plants',
    'Barley':'Harvested production in EU standard humidity (1000 t)_Barley',
    'Oranges':'Harvested production in EU standard humidity (1000 t)_Citrus fruits',
    'Wheat':'Harvested production in EU standard humidity (1000 t)_Wheat and spelt',
    'Seed cotton, unginned':'Harvested production in EU standard humidity (1000 t)_Cotton seed',
    
    'True hemp, raw or retted':'Harvested production in EU standard humidity (1000 t)_Hemp',
    'Linseed':'Harvested production in EU standard humidity (1000 t)_Linseed (oilflax)',
    'Olives':'Harvested production in EU standard humidity (1000 t)_Olives',
    'Potatoes':'Harvested production in EU standard humidity (1000 t)_Potatoes (including seed potatoes)',
    'Sorghum':'Harvested production in EU standard humidity (1000 t)_Sorghum',
    'Soya beans':'Harvested production in EU standard humidity (1000 t)_Soya',
    'Sugar beet':'Harvested production in EU standard humidity (1000 t)_Sugar beet (excluding seed)',
    'Sunflower seed':'Harvested production in EU standard humidity (1000 t)_Sunflower seed',
    'Lupins':'Harvested production in EU standard humidity (1000 t)_Sweet lupins',
    'Unmanufactured tobacco':'Harvested production in EU standard humidity (1000 t)_Tobacco',
    'Triticale':'Harvested production in EU standard humidity (1000 t)_Triticale',
    'Rye':'Harvested production in EU standard humidity (1000 t)_Rye',
    'Oats':'Harvested production in EU standard humidity (1000 t)_Oats',
    'Rice':'Harvested production in EU standard humidity (1000 t)_Rice',
    'Peas, dry':'Harvested production in EU standard humidity (1000 t)_Field peas',
    'Other fibre crops, raw, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Other fibre crops n.e.c.',
    'Other oil seeds, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Other oilseed crops n.e.c.',
    'Grapes':'Harvested production in EU standard humidity (1000 t)_Grapes',
    
    'Broad beans and horse beans, dry':'Harvested production in EU standard humidity (1000 t)_Broad and field beans',
    'Flax, raw or retted':'Harvested production in EU standard humidity (1000 t)_Fibre crops',
    
    ('Watermelons','Strawberries'):'Harvested production in EU standard humidity (1000 t)_Fruits, berries and nuts (excluding citrus fruits, grapes and strawberries)',
    'Maize (corn)':'Harvested production in EU standard humidity (1000 t)_Green maize',
    'Hop cones':'Harvested production in EU standard humidity (1000 t)_Hops',
    'Other beans, green':'Harvested production in EU standard humidity (1000 t)_Leguminous plants harvested green',
   
    'Millet':'Harvested production in EU standard humidity (1000 t)_Other cereals n.e.c. (buckwheat, millet, canary seed, etc.)',
    'Rape or colza seed':'Harvested production in EU standard humidity (1000 t)_Rape, turnip rape, sunflower seeds and soya',
    'Cereals n.e.c.':'Harvested production in EU standard humidity (1000 t)_Spring cereal mixtures (mixed grain other than maslin)'
} # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称

fao_target_path = '校对总量全（新）/欧盟/FAO/农作物/' # 规范化后的fao数据的保存路径
fao_data_path = 'eurostat/农作物_ok/FAO/FAOSTAT_data_en_1-28-2024.csv'  # fao原始数据的存放路径
data_path = 'eurostat/农作物_ok/' # 需要校对的国家数据存放路径
r_path = '校对总量全（新）/欧盟/FAO/比例县.csv' # 历史比例存放路径
data_fao_ok = '校对总量全（新）/欧盟/' # 校对完成后的数据存放路径

# 校对国家的开始与结束年份
year_begin = 2021
year_end = 2021

# 执行
---

In [5]:
# fao官网在维护，这里用之前的数据，所以要合并，农作物
fao_all = pd.DataFrame()
for y in range(year_begin,year_end+1):
    fao_y = pd.read_csv("D:/中科院数据下载/faostat_HGU/Production/Crops and livestock products/Production_Quantity/Crops_primary/"+str(y)+'.csv')
    data = fao_y[fao_y['Area']=='Australia']
    fao_all = pd.concat([fao_all,data],axis=0)
    fao_all.to_csv(fao_data_path,index=False,encoding='utf-8-sig')

In [8]:
# 函数一
# 规范化fao数据
fao_data = pd.read_csv(fao_data_path)
fao_standard(fao_data,params,fao_target_path)

# 函数二
r = pd.DataFrame()
for y in range(year_begin,year_end+1):
    data = pd.read_excel(data_path+str(y)+".xlsx")
    # data = pd.read_csv(data_path+str(y)+".csv")

    data_fao = pd.read_csv(fao_target_path+str(y)+".csv")
    r_tmp = cal_r(data,data_fao,y,data_fao_item="Item")

    r = pd.concat([r,r_tmp],axis=0)
r.to_csv(r_path,index=False,encoding='utf-8-sig')

# 函数三
# 进行校对
for y in range(year_begin,year_end+1):
    data = pd.read_excel(data_path+str(y)+".xlsx")
    # data = pd.read_csv(data_path+str(y)+".csv")
    # data = unit_conversion(data,params_unit) # 单位转化
    data_fao = pd.read_csv(fao_target_path+str(y)+".csv")
    data_ok = single_proofread_r(data,data_fao,r,y,data_fao_item="Item")

    data_ok.to_csv(data_fao_ok+str(y)+".csv",index=False,encoding="utf-8-sig")


Index(['Level', 'Code', 'Municipality', 'Pineapple*', 'Alfalfa hayed',
       'Herbaceous cotton (seed)', 'Garlic', 'Peanuts (in shell)', 'Rice',
       'Oats (grain)', 'Sweet potatoes', 'Batata-inglesa', 'Sugarcane',
       'Forage cane', 'Onion', 'Rye (grain)', 'Barley (grain)', 'Peas (grain)',
       'Fava beans', 'Beans (beans)', 'Tobacco(leaf)', 'Sunflower (grain)',
       'Jute (fiber)', 'Flax (seed)', 'Mallow (fiber)', 'Mormon(bag)',
       'Cassava', 'Watermelon', 'Melon', 'Corn (in grain)', 'Rami (fiber)',
       'Soybeans', 'Sorghum (grain)', 'Tomato', 'Wheat (grain)',
       'Triticale (grain)', 'Abacate', 'Tree cotton (seed)', 'Acai berry',
       'Olive', 'Banana (bunch)', 'Rubber (coagulated latex)',
       'Rubber (liquid latex)', 'Cocoa beans', 'Coffee (beans) Total',
       'Coffee (beans) Arabica', 'Coffee (beans) Canephora', 'Cashew',
       'Persimmon', 'Cashew nuts', 'Tea(leafy green)', 'Coco-da-baia',
       'Palm oil(coconut bunch)', 'Yerba mate (green leaf)', 'F

In [7]:
import pandas as pd

pd.options.mode.chained_assignment = None  # 默认为'warn'

# 欧盟农作物专属执行
data2022 = pd.read_excel('校对总量全（新）/欧盟/countrys.xlsx')
countrys = data2022['Country'].unique()

# 要保留在FAO不存在而原始存在的
# 
import chardet  
data_all = pd.DataFrame()
fao_data_all = pd.read_csv(fao_data_path)
for i in countrys:
    # 函数一
    # 规范化fao数据
    fao_data = fao_data_all[fao_data_all['Area']==i]
    if len(fao_data)==0:
        print('这个国家在FAO数据中不存在：'+i) 
        continue
    
    fao_standard(fao_data,params,fao_target_path)

    # 函数三
    # 进行校对
    for y in range(year_begin,year_end+1):
     
        with open(data_path+str(y)+".csv", 'rb') as file:  
            result = chardet.detect(file.read())  # 或者file.read(10000)来读取部分文件进行检测  
        encoding = result['encoding']  
       
        data = pd.read_csv(data_path+str(y)+".csv", encoding=encoding)  

        data = data.dropna(subset=['county'])

        data = data[data['country_label']==i]
        data.iloc[:,7:] *= 1000

        data_fao = pd.read_csv(fao_target_path+str(y)+".csv")
        
        data_ok = single_proofread_r(data,data_fao)
        data_all = pd.concat([data_all,data_ok],axis=0)
data_all.to_csv(data_fao_ok+str(y)+".csv",index=False,encoding="utf-8-sig")


该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Barley，校对成功，校对后为：1514491.0，fao值为1514491.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Broad and field beans，校对成功，校对后为：7100.0，fao值为7100.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Spring cereal mixtures (mixed grain other than maslin)，校对成功，校对后为：0.0，fao值为0.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Fibre crops，校对成功，校对后为：0.0，fao值为0.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Grapes，校对成功，校对后为：439000.0000000001，fao值为439000.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Hops，校对成功，校对后为：171.99999999999997，fao值为172.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Linseed (oilflax)，校对成功，校对后为：0.0，fao值为0.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Sweet lupins，校对成功，校对后为：0.0，fao值为0.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Green maize，校对成功，校对后为：1292745.0，fao值为1292745.0
该国该种类：9
国家：['Austria']的种类：Harvested production (1000 t)_Other cereals n.

In [8]:
countrys

array(['Austria', 'Belgium', 'Bulgaria', 'Czechia', 'Cyprus', 'Germany',
       'Denmark', 'Spain', 'Greece', 'Estonia', 'Finland', 'France',
       'Netherlands', 'Ireland', 'Italy', 'Hungary', 'Croatia', 'Malta',
       'Lithuania', 'Luxembourg', 'Latvia', 'Poland', 'Romania',
       'Portugal', 'Slovenia', 'Slovakia', 'Sweden', 'United Kingdom'],
      dtype=object)

In [31]:
import pandas as pd
data = pd.read_csv('校对总量全（新）/欧盟/农作物/2022.csv')
data.head()

,country,country_label,state,state_label,county,county_label,year,"Area (cultivation/harvested/production) (1000 ha)_Aromatic, medicinal and culinary plants",Area (cultivation/harvested/production) (1000 ha)_Barley,Area (cultivation/harvested/production) (1000 ha)_Broad and field beans,...,Main area (1000 ha)_Other root crops n.e.c.,Main area (1000 ha)_Permanent crops,Main area (1000 ha)_Permanent grassland,Main area (1000 ha)_Plants harvested green from arable land,Main area (1000 ha)_Potatoes (including seed potatoes),Main area (1000 ha)_Root crops,Main area (1000 ha)_Seeds and seedlings,Main area (1000 ha)_Sugar beet (excluding seed),Main area (1000 ha)_Utilised agricultural area,标记
0,AT,Austria,NaN,NaN,NaN,NaN,2022,3.75,122.55,5.54,...,0.08,66.87,1209.98,225.15,21.44,55.51,0.28,33.99,2599.51,NaN
1,AT,Austria,AT1,Ostösterreich,NaN,NaN,2022,3.12,69.46,3.50,...,0.03,46.68,188.67,104.15,18.37,44.78,0.23,26.38,1069.06,NaN
2,AT,Austria,AT1,Ostösterreich,AT11,Burgenland (AT),2022,0.17,6.62,0.36,...,0.01,13.27,12.66,16.51,1.32,3.27,0.00,1.94,183.13,NaN
3,AT,Austria,AT1,Ostösterreich,AT12,Niederösterreich,2022,2.92,62.48,3.13,...,0.02,32.61,175.23,87.44,16.99,41.35,0.23,24.33,880.54,NaN
4,AT,Austria,AT1,Ostösterreich,AT13,Wien,2022,0.02,0.35,0.02,...,0.00,0.79,0.78,0.21,0.05,0.16,0.00,0.11,5.40,NaN


In [32]:
data.iloc[:,7:] *=1000

In [33]:
data.to_csv('校对总量全（新）/欧盟/2022.csv',encoding='utf-8-sig')

databaseName=chatlist-history&create_time=create_date&filter=%7B%E6%80%BB%E9%87%8F%3A%7B%7D%2C%E6%8B%8D%E7%85%A7%E6%90%9C%E9%A2%98%3A%7B%20rules%3A%20%5B%7B%20type%3A%20'exists'%2C%20field%3A%20'imgurl'%20%7D%5D%20%7D%2C%E6%96%87%E5%AD%97%E6%90%9C%E9%A2%98%3A%7B%20rules%3A%20%5B%7B%20type%3A%20'notExists'%2C%20field%3A%20'imgurl'%20%7D%2C%20%7B%20type%3A%20'notEquals'%2C%20field%3A%20'question'%2C%20value%3A%20''%20%7D%5D%20%7D%2C%E8%81%8A%E5%A4%A9%3A%7B%20rules%3A%20%5B%7B%20type%3A%20'equals'%2C%20field%3A%20'question'%2C%20value%3A%20''%20%7D%5D%20%7D%20%7D&StartTime=&EndTime=&TimeRange=month&TimeInterval=day&filter=%7B%E9%97%AE%E6%99%BA%E9%80%9A%3A%7Brules%3A%5B%7Btype%3A'equals'%2Cfield%3A'model'%2Cvalue%3A'%E9%97%AE%E6%99%BA%E9%80%9A-4'%7D%5D%7D%2C%E6%96%87%E5%BF%834%3A%7Brules%3A%5B%7Btype%3A'equals'%2Cfield%3A'model'%2Cvalue%3A'BAIDU-4'%7D%5D%7D%2CGPT-4%3A%7Brules%3A%5B%7Btype%3A'equals'%2Cfield%3A'model'%2Cvalue%3A'GPT-4'%7D%5D%7D%7D
